# YOLO26n 사람·장애물·보행 신호 통합 학습

런타임 → 런타임 유형 변경 → **GPU(T4)** 를 고른 뒤 위에서부터 실행한다.

- 데이터: `ml/scripts/remap_labels.py` → `split_dataset.py` 로 만든 폴더를 `dataset.zip` 으로 묶어 Drive 에 올린다.
- 결과(가중치)는 Drive 에 저장되어 세션이 끊겨도 이어서 학습할 수 있다. 가중치·데이터는 커밋하지 않는다.

## 1. 설정

In [ ]:
# Drive 안 경로와 학습 값. 여기만 바꾼다.
DRIVE_ROOT = '/content/drive/MyDrive/sait-gil'
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'
RUNS_DIR = f'{DRIVE_ROOT}/runs'
RUN_NAME = 'v1'
LOCAL_DATASET = '/content/dataset'

BASE_WEIGHTS = 'yolo26n.pt'  # COCO 사전학습 가중치에서 시작
EPOCHS = 100
IMGSZ = 640
PATIENCE = 20                # 이만큼 개선이 없으면 조기 종료
SAVE_PERIOD = 5

SMOKE_TEST = True            # 처음엔 True: 데이터 일부·짧은 학습으로 파이프라인만 확인
SMOKE_EPOCHS = 10
SMOKE_FRACTION = 0.1

## 2. Drive 연결 · 데이터 복사 · 설치
Drive 에서 바로 읽으면 매우 느리므로 Colab 로컬 디스크로 풀어 쓴다.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!cp "{DATASET_ZIP}" /content/dataset.zip
!unzip -q -o /content/dataset.zip -d "{LOCAL_DATASET}"
!pip install -q -U ultralytics
!nvidia-smi

## 3. 학습
세션이 끊겼으면 이 셀 대신 아래 '이어서 학습' 셀을 실행한다.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_WEIGHTS)
model.train(
    data=f'{LOCAL_DATASET}/data.yaml',
    epochs=SMOKE_EPOCHS if SMOKE_TEST else EPOCHS,
    fraction=SMOKE_FRACTION if SMOKE_TEST else 1.0,
    imgsz=IMGSZ,
    batch=-1,  # GPU 메모리에 맞게 자동
    patience=PATIENCE,
    project=RUNS_DIR,
    name=f'{RUN_NAME}-smoke' if SMOKE_TEST else RUN_NAME,
    save_period=SAVE_PERIOD,
    cache='disk',
)

### 이어서 학습 (세션이 끊겼을 때)

In [ ]:
from ultralytics import YOLO

YOLO(f'{RUNS_DIR}/{RUN_NAME}/weights/last.pt').train(resume=True)

## 4. 평가
전체 mAP 보다 **클래스별 recall** 을 본다. 놓치는 것(미탐)이 더 위험하다: person, ped_signal_*, bollard 우선.
빨강↔초록 오판은 run 폴더의 `confusion_matrix.png` 로 확인한다.

In [ ]:
from ultralytics import YOLO

best = YOLO(f'{RUNS_DIR}/{RUN_NAME}/weights/best.pt')
metrics = best.val(data=f'{LOCAL_DATASET}/data.yaml', split='test')
# test 에 박스가 있는 클래스만 나온다. 빠진 클래스는 test 데이터부터 채운다.
for i, idx in enumerate(metrics.box.ap_class_index):
    p, r, ap50, _ = metrics.box.class_result(i)
    print(f'{idx:>2} {best.names[idx]:<18} P={p:.3f} R={r:.3f} mAP50={ap50:.3f}')

직접 찍은 영상(얼굴 비식별 처리한 것)으로 눈으로 확인한다. 영상은 Drive 에 올려 둔다.

In [ ]:
TEST_VIDEO = f'{DRIVE_ROOT}/test_videos/sample.mp4'
best.predict(source=TEST_VIDEO, save=True, project=RUNS_DIR, name=f'{RUN_NAME}-predict')

## 5. LiteRT(.tflite) 변환
int8 이 작고 빠르다. 정확도가 많이 떨어지면 `half=True`(FP16) 로 바꿔 비교한다.
기기에서 느리면 `IMGSZ` 를 480·320 으로 낮춰 다시 내보낸다(멀리 있는 신호는 놓치기 쉬워짐).
만든 `.tflite` 는 `frontend/android/app/src/main/assets/models/` 로 복사한다(커밋 X).

In [ ]:
tflite_path = best.export(
    format='tflite', imgsz=IMGSZ, int8=True, data=f'{LOCAL_DATASET}/data.yaml'
)
print(tflite_path)